# Lab | Hyperparameter Tuning

**Dataset:** Spaceship Titanic  
**Objetivo:** Optimizar el rendimiento del mejor modelo de clasificación mediante GridSearchCV  
**Tipo de problema:** Clasificación binaria — predecir si un pasajero fue transportado

## Importar librerías

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluación y tuning
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Librerías importadas correctamente')

## Cargar los datos

In [ ]:
# Cargamos el dataset de Spaceship Titanic
spaceship = pd.read_csv('https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv')

print(f'Shape: {spaceship.shape}')
spaceship.head()

## Feature Engineering

Realizamos el mismo preprocesamiento que en labs anteriores:
- Eliminamos columnas no informativas
- Gestionamos nulos
- Encoding de variables categóricas
- Escalado con MinMaxScaler
- Selección de features

In [ ]:
# Revisión inicial del dataset
print('Tipos de datos:')
print(spaceship.dtypes)
print('\nValores nulos por columna:')
print(spaceship.isnull().sum())

In [ ]:
# Eliminamos columnas que no aportan información predictiva
# PassengerId y Name son identificadores únicos
# Cabin tiene demasiadas categorías para encodear directamente
spaceship.drop(columns=['PassengerId', 'Name', 'Cabin'], inplace=True)

print(f'Shape tras eliminar columnas: {spaceship.shape}')

In [ ]:
# Gestionamos nulos — rellenamos con la moda para categóricas y la mediana para numéricas
cat_cols = spaceship.select_dtypes(include='object').columns.tolist()
num_cols = spaceship.select_dtypes(include='number').columns.tolist()

for col in cat_cols:
    spaceship[col].fillna(spaceship[col].mode()[0], inplace=True)

for col in num_cols:
    spaceship[col].fillna(spaceship[col].median(), inplace=True)

print(f'Valores nulos tras imputación: {spaceship.isnull().sum().sum()}')

In [ ]:
# Encoding de variables categóricas con get_dummies
spaceship = pd.get_dummies(spaceship, drop_first=True, dtype=int)

print(f'Shape tras encoding: {spaceship.shape}')
spaceship.head()

## Feature Scaling y Train/Test Split

Aplicamos MinMaxScaler igual que en el notebook de clase.  
El scaler se ajusta solo con X_train para evitar data leakage.

In [ ]:
# Separamos features y target
X = spaceship.drop(columns='Transported')
y = spaceship['Transported']

# Split 80/20 con stratify para mantener la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

In [ ]:
# Escalado con MinMaxScaler — fit solo en train
normalizer = MinMaxScaler()
normalizer.fit(X_train)

# Transformamos train y test
X_train_norm = pd.DataFrame(normalizer.transform(X_train), columns=X_train.columns)
X_test_norm  = pd.DataFrame(normalizer.transform(X_test),  columns=X_test.columns)

print('Escalado completado')
print(X_train_norm.describe().loc[['min', 'max']].round(2))

## Modelos baseline — comparación previa al tuning

Antes de hacer tuning, entrenamos varios modelos con parámetros por defecto  
para identificar cuál es el más prometedor.

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_norm, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test_norm))
print(f'Logistic Regression — Accuracy: {acc_lr:.4f}')

In [ ]:
# --- Decision Tree ---
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_norm, y_train)
acc_dt = accuracy_score(y_test, dt.predict(X_test_norm))
print(f'Decision Tree — Accuracy: {acc_dt:.4f}')

In [ ]:
# --- KNN ---
knn = KNeighborsClassifier()
knn.fit(X_train_norm, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test_norm))
print(f'KNN — Accuracy: {acc_knn:.4f}')

In [ ]:
# --- Random Forest ---
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_norm, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test_norm))
print(f'Random Forest — Accuracy: {acc_rf:.4f}')

In [ ]:
# --- Gradient Boosting ---
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_norm, y_train)
acc_gb = accuracy_score(y_test, gb.predict(X_test_norm))
print(f'Gradient Boosting — Accuracy: {acc_gb:.4f}')

In [ ]:
# Tabla comparativa de modelos baseline
baseline = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Decision Tree', 'KNN', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [acc_lr, acc_dt, acc_knn, acc_rf, acc_gb]
})

baseline.sort_values('Accuracy', ascending=False).reset_index(drop=True)

## Grid Search — Hyperparameter Tuning

Usamos GridSearchCV con el mejor modelo del baseline.  
Grid Search prueba todas las combinaciones posibles del grid de hiperparámetros  
y selecciona la que mejor resultado obtiene en validación cruzada.

Los hiperparámetros que optimizamos en Random Forest son:
- **n_estimators**: número de árboles en el bosque
- **max_depth**: profundidad máxima de cada árbol
- **min_samples_split**: número mínimo de muestras para dividir un nodo

In [ ]:
# Definimos el grid de hiperparámetros a explorar
grid = {
    'n_estimators':     [50, 100, 200],
    'max_depth':        [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

In [ ]:
# Creamos la instancia del modelo base
rf_base = RandomForestClassifier(random_state=42)

In [ ]:
# Configuramos GridSearchCV con validación cruzada de 5 folds
model = GridSearchCV(
    estimator=rf_base,
    param_grid=grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Entrenamos GridSearchCV — prueba todas las combinaciones del grid
model.fit(X_train_norm, y_train)

In [ ]:
# Mejores hiperparámetros encontrados por GridSearchCV
print(f'Mejores parámetros: {model.best_params_}')
print(f'Mejor CV Accuracy:  {model.best_score_:.4f}')

In [ ]:
# Recuperamos el mejor modelo con los mejores parámetros
best_model = model.best_estimator_

## Evaluación del modelo final

Evaluamos el mejor modelo obtenido por GridSearchCV sobre el conjunto de test.

In [ ]:
# Predicciones del mejor modelo sobre el test
y_pred_best = best_model.predict(X_test_norm)

print('=== EVALUACIÓN DEL MEJOR MODELO (GridSearchCV) ===')
print(classification_report(y_test, y_pred_best, target_names=['No transportado (0)', 'Transportado (1)']))

In [ ]:
# Matriz de confusión del mejor modelo
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No transportado', 'Transportado'],
            yticklabels=['No transportado', 'Transportado'])
plt.title('Matriz de Confusión — Mejor modelo (GridSearchCV)')
plt.tight_layout()
plt.show()

## Comparación baseline vs modelo tuneado

In [ ]:
# Comparamos el accuracy del Random Forest baseline vs el tuneado con GridSearchCV
acc_tuned = accuracy_score(y_test, y_pred_best)

print(f'Random Forest baseline:       {acc_rf:.4f}')
print(f'Random Forest tuned (GridCV): {acc_tuned:.4f}')
print(f'Mejora: {(acc_tuned - acc_rf)*100:.2f} puntos porcentuales')
print(f'\nMejores parámetros encontrados: {model.best_params_}')
print('\nNo garantizamos que estos sean los parámetros óptimos absolutos —')
print('solo los mejores de entre los que hemos explorado en el grid.')